<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 🖥️ Donanım Gereksinimleri Analizi

Bu bölüm, GPT-2 modellerinin donanım gereksinimlerini (VRAM, RAM, CPU) hesaplar.

In [1]:
import pandas as pd

# ========================================
# 📊 MODEL GEREKSİNİMLERİ HESAPLAMA
# ========================================

def calculate_model_requirements(emb_dim, n_layers, n_heads, vocab_size=50257, context_length=1024):
    """
    Model parametrelerini ve bellek gereksinimlerini hesapla
    """
    
    # 1. Token Embedding
    tok_emb = vocab_size * emb_dim
    
    # 2. Positional Embedding
    pos_emb = context_length * emb_dim
    
    # 3. Attention (QKV + OutProj)
    qkv = 3 * (emb_dim * emb_dim)
    out_proj = emb_dim * emb_dim
    att_params = qkv + out_proj
    
    # 4. Feed Forward (4x expansion)
    ffn_up = emb_dim * (emb_dim * 4)
    ffn_down = (emb_dim * 4) * emb_dim
    ffn_params = ffn_up + ffn_down
    
    # 5. LayerNorm
    ln_params = 2 * emb_dim
    
    # Bir transformer block
    block_params = att_params + ffn_params + ln_params
    
    # Tüm block'lar
    total_block_params = block_params * n_layers
    
    # Output Head
    out_head = emb_dim * vocab_size
    
    # Final LayerNorm
    final_ln = 2 * emb_dim
    
    # TOPLAM PARAMETRELER
    total_params = tok_emb + pos_emb + total_block_params + final_ln + out_head
    
    # --- MEMORY HESAPLAMA ---
    
    # Inference için (sadece model)
    model_memory_fp32_gb = (total_params * 4) / (1024**3)  # 4 bytes per FP32
    model_memory_fp16_gb = (total_params * 2) / (1024**3)  # 2 bytes per FP16
    model_memory_bf16_gb = (total_params * 2) / (1024**3)  # 2 bytes per BF16
    
    # Training için (model + gradients + optimizer)
    # Gradients: same as model
    # Optimizer (AdamW): 2x model size (momentum + variance)
    train_memory_fp32_gb = (total_params * 4 * 4) / (1024**3)  # model + grad + 2x optimizer state
    train_memory_fp16_gb = (total_params * 2 * 4) / (1024**3)
    
    # KV Cache (inference için ek bellek)
    head_dim = emb_dim // n_heads
    kv_cache_per_layer_gb = (2 * context_length * n_layers * n_heads * head_dim * 2) / (1024**3)
    
    return {
    "total_params": total_params,
    "model_memory_fp32_gb": model_memory_fp32_gb,
    "model_memory_fp16_gb": model_memory_fp16_gb,
    "model_memory_bf16_gb": model_memory_bf16_gb,
    "train_memory_fp32_gb": train_memory_fp32_gb,
    "train_memory_fp16_gb": train_memory_fp16_gb,
    "kv_cache_gb": kv_cache_per_layer_gb,
    "inference_total_fp16_gb": model_memory_fp16_gb + kv_cache_per_layer_gb
    }

# Model konfigürasyonları
model_configs = {
    "GPT-2 Small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "GPT-2 Medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "GPT-2 Large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "GPT-2 XL (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Her model için gereksinimleri hesapla
results = []
for model_name, config in model_configs.items():
    req = calculate_model_requirements(config['emb_dim'], config['n_layers'], config['n_heads'])
    results.append({
        "Model": model_name,
        "Parametre": f"{req['total_params']/1e6:.0f}M",
        "Model (FP32)": f"{req['model_memory_fp32_gb']:.2f} GB",
        "Model (FP16)": f"{req['model_memory_fp16_gb']:.2f} GB",
        "Inference+KV (FP16)": f"{req['inference_total_fp16_gb']:.2f} GB",
        "Training (FP16)": f"{req['train_memory_fp16_gb']:.2f} GB",
    })

df = pd.DataFrame(results)
print("=" * 100)
print("📊 MODEL DONANIM GEREKSİNİMLERİ TABLOSU")
print("=" * 100)
print(df.to_string(index=False))
print("=" * 100)

📊 MODEL DONANIM GEREKSİNİMLERİ TABLOSU
              Model Parametre Model (FP32) Model (FP16) Inference+KV (FP16) Training (FP16)
 GPT-2 Small (124M)      163M      0.61 GB      0.30 GB             0.34 GB         1.21 GB
GPT-2 Medium (355M)      406M      1.51 GB      0.76 GB             0.85 GB         3.03 GB
 GPT-2 Large (774M)      838M      3.12 GB      1.56 GB             1.74 GB         6.24 GB
   GPT-2 XL (1558M)     1637M      6.10 GB      3.05 GB             3.34 GB        12.20 GB


In [2]:
# ========================================
# 🖥️ ÖNERİLEN DONANIM
# ========================================

print("\n" + "=" * 100)
print("🖥️ ÖNERİLEN DONANIM PER MODEL")
print("=" * 100)

hardware_recommendations = []

for model_name, config in model_configs.items():
    req = calculate_model_requirements(config['emb_dim'], config['n_layers'], config['n_heads'])
    
    # Inference için önerilen VRAM
    inf_vram = req['inference_total_fp16_gb']
    if inf_vram <= 6:
        vram_inf = "6GB+"
    elif inf_vram <= 10:
        vram_inf = "10GB+"
    elif inf_vram <= 16:
        vram_inf = "16GB+"
    elif inf_vram <= 24:
        vram_inf = "24GB+"
    else:
        vram_inf = "40GB+"
    
    # Training için önerilen VRAM
    train_vram = req['train_memory_fp16_gb']
    if train_vram <= 8:
        vram_train = "8GB (sınırlı)"
    elif train_vram <= 16:
        vram_train = "16GB+"
    elif train_vram <= 24:
        vram_train = "24GB+"
    elif train_vram <= 40:
        vram_train = "40GB+"
    else:
        vram_train = "80GB+ (A100/H100)"
    
    # Önerilen RAM
    ram_suggested = "16GB+"
    
    hardware_recommendations.append({
        "Model": model_name,
        "VRAM (Inference)": vram_inf,
        "VRAM (Training)": vram_train,
        "Sistem RAM": ram_suggested
    })

df_hw = pd.DataFrame(hardware_recommendations)
print(df_hw.to_string(index=False))
print("=" * 100)


🖥️ ÖNERİLEN DONANIM PER MODEL
              Model VRAM (Inference) VRAM (Training) Sistem RAM
 GPT-2 Small (124M)             6GB+   8GB (sınırlı)      16GB+
GPT-2 Medium (355M)             6GB+   8GB (sınırlı)      16GB+
 GPT-2 Large (774M)             6GB+   8GB (sınırlı)      16GB+
   GPT-2 XL (1558M)             6GB+           16GB+      16GB+


---

## 📋 Özet

### Donanım Gereksinimleri:

| Model | Parametre | Inference VRAM | Training VRAM | Uygun GPU'lar |
|-------|-----------|----------------|---------------|----------------|
| GPT-2 Small | 124M | ~1.5 GB | ~6 GB | GTX 1660, RTX 3060 |
| GPT-2 Medium | 355M | ~4 GB | ~18 GB | RTX 3070, RTX 3080 |
| GPT-2 Large | 774M | ~8 GB | ~40 GB | RTX 3090, RTX 4090 |
| GPT-2 XL | 1558M | ~16 GB | ~80 GB | A100, H100 |

### 💡 Öneriler:

1. **Başlangıç için**: 124M model, 8GB VRAM yeterli
2. **Orta seviye**: 355M model, 10GB+ VRAM
3. **İleri seviye**: 774M+ model, 24GB+ VRAM gerekli
4. **Training**: Her zaman FP16/BF16 kullanın (2x memory tasarrufu)

# FLOPS Analysis

- FLOPs (Floating Point Operations Per Second) measure the computational complexity of neural network models by counting the number of floating-point operations executed
- High FLOPs indicate more intensive computation and energy consumption

In [3]:
#%pip install -r requirements-extra.txt

In [4]:
%pip install thop

In [5]:
from importlib.metadata import version

pkgs = [
    "thop",
    "torch",
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

thop version: 0.1.1-2209072238
torch version: 2.10.0+cu128


&nbsp;
# Simple benchmark with fixed batch size

- forward pass only

In [6]:
#for colab 
import torch
from torch import nn
from thop import profile
import sys
# This file collects all the relevant code that we covered thus far
# throughout Chapters 2-4.
# This file can be run as a standalone script.

import tiktoken
from torch.utils.data import Dataset, DataLoader

#####################################
# Chapter 2
#####################################


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


#####################################
# Chapter 3
#####################################
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)  # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec


#####################################
# Chapter 4
#####################################
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)   # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus only on the last time step
        # (batch, n_token, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Get the idx of the vocab entry with the highest logits value
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx


def main():
    GPT_CONFIG_124M = {
        "vocab_size": 50257,     # Vocabulary size
        "context_length": 1024,  # Context length
        "emb_dim": 768,          # Embedding dimension
        "n_heads": 12,           # Number of attention heads
        "n_layers": 12,          # Number of layers
        "drop_rate": 0.1,        # Dropout rate
        "qkv_bias": False        # Query-Key-Value bias
    }

    torch.manual_seed(123)
    model = GPTModel(GPT_CONFIG_124M)
    model.eval()  # disable dropout

    start_context = "Hello, I am"

    tokenizer = tiktoken.get_encoding("gpt2")
    encoded = tokenizer.encode(start_context)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)

    print(f"\n{50*'='}\n{22*' '}IN\n{50*'='}")
    print("\nInput text:", start_context)
    print("Encoded input text:", encoded)
    print("encoded_tensor.shape:", encoded_tensor.shape)

    out = generate_text_simple(
        model=model,
        idx=encoded_tensor,
        max_new_tokens=10,
        context_size=GPT_CONFIG_124M["context_length"]
    )
    decoded_text = tokenizer.decode(out.squeeze(0).tolist())

    print(f"\n\n{50*'='}\n{22*' '}OUT\n{50*'='}")
    print("\nOutput:", out)
    print("Output length:", len(out[0]))
    print("Output text:", decoded_text)


if __name__ == "__main__":
    main()



                      IN

Input text: Hello, I am
Encoded input text: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


                      OUT

Output: tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267,
         49706, 43231, 47062, 34657]])
Output length: 14
Output text: Hello, I am Featureiman Byeswickattribute argue logger Normandy Compton analogous


In [7]:
import torch
from thop import profile
import sys

# Bilimsel gösterimi kapat
torch.set_printoptions(sci_mode=False)

#for local
# Add path to local gpt.py file
#sys.path.append('../01_main-chapter-code')
#from gpt import GPTModel


BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25}
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 2
input_tensor = torch.randint(0, 50257, (batch_size, 1024)).to(device)

for size in model_configs:
    BASE_CONFIG.update(model_configs[size])

    model = GPTModel(BASE_CONFIG).bfloat16()
    model.to(device)

    # MACS = multiply-accumulate operations
    # MACS are typically counted as two FLOPS (one multiply and one accumulate)
    macs, params = profile(model, inputs=(input_tensor,), verbose=False)
    flops = 2*macs
    print(f"{size:18}: {flops:.1e} FLOPS")

    del model
    torch.cuda.empty_cache()




gpt-small (124M)  : 5.1e+11 FLOPS
gpt-medium (355M) : 1.4e+12 FLOPS
gpt-large (774M)  : 3.2e+12 FLOPS
gpt-xl (1558M)    : 6.4e+12 FLOPS


&nbsp;
# Simple benchmark with automatic batch size finding

- forward pass only

In [8]:
for size in model_configs:
    print(f"\nProcessing {size}")
    config = BASE_CONFIG.copy()
    config.update(model_configs[size])

    min_batch_size = 1
    max_batch_size = None
    max_possible_batch_size = 4096

    while min_batch_size <= max_possible_batch_size:
        batch_size = (min_batch_size + max_possible_batch_size) // 2
        try:
            input_tensor = torch.randint(
                0, config["vocab_size"],
                (batch_size, config["context_length"]),
                device=device
            )

            model = GPTModel(config).bfloat16().to(device)

            # MACS = multiply-accumulate operations
            # MACS are typically counted as two FLOPS (one multiply and one accumulate)
            macs, params = profile(model, inputs=(input_tensor,), verbose=False)
            flops = 2 * macs
            print(f"  Batch size {batch_size}: {flops:.1e} FLOPS")

            # If successful, try a larger batch size
            min_batch_size = batch_size + 1
            max_batch_size = batch_size

            # Clean up
            del model, input_tensor
            torch.cuda.empty_cache()

        except RuntimeError as e:
            if "out of memory" in str(e):
                # Try smaller batch size
                max_possible_batch_size = batch_size - 1

                # Clean up
                try:
                    del model, input_tensor
                    torch.cuda.empty_cache()
                except NameError:
                    pass
            else:
                raise e


Processing gpt-small (124M)
  Batch size 64: 1.6e+13 FLOPS
  Batch size 96: 2.4e+13 FLOPS
  Batch size 112: 2.8e+13 FLOPS
  Batch size 120: 3.0e+13 FLOPS
  Batch size 124: 3.1e+13 FLOPS
  Batch size 126: 3.2e+13 FLOPS

Processing gpt-medium (355M)
  Batch size 64: 4.6e+13 FLOPS
  Batch size 96: 6.9e+13 FLOPS

Processing gpt-large (774M)
  Batch size 64: 1.0e+14 FLOPS
  Batch size 66: 1.0e+14 FLOPS
  Batch size 67: 1.1e+14 FLOPS

Processing gpt-xl (1558M)
  Batch size 64: 2.0e+14 FLOPS


&nbsp;
# Benchmark with automatic batch size finding and Model FLOP Utilization (MFU)

- Model FLOPs Utilization (MFU) explanation from the <a href="https://arxiv.org/abs/2204.02311">PaLM paper</a>

> We propose a new metric for efficiency that is implementation-independent and permits a cleaner comparison of system efficiency, called model FLOPs utilization (MFU). This is the ratio of the observed throughput (tokens-per-second) relative to the theoretical maximum throughput of a system operating at peak FLOPs. Crucially, the "theoretical maximum" throughput only accounts for the required operations to compute the forward+backward passes, and not rematerialization.


$$\text{MFU} = \frac{\text{Observed Tokens per Second}}{\text{Theoretical Max Tokens per Second}}$$

where

$$\text{Theoretical Max Tokens per Second} = \frac{\text{Max FLOPs per Second}}{\text{Total FLOPs per Token}}$$

and

$$\text{Tokens per Second} = \frac{\text{Batch Size} \times \text{Sequence Length}}{\text{Total Time}}$$

- forward and backward pass

In [9]:
# Theoretical max flops per second provided by the GPU manufacturer

flops_per_second = {
    # https://www.techpowerup.com/gpu-specs/h100-pcie-80-gb.c3899
    "H100": {
        torch.float32: 51.22e12,  # 51.22 TFLOPs for FP32 on NVIDIA H100
        torch.float16: 204.9e12,  # 204.9 TFLOPs for FP16 on NVIDIA H100
        torch.bfloat16: 204.9e12
    },
    # https://www.techpowerup.com/gpu-specs/l4.c4091
    "L4": {
        torch.float32: 30.29e12,  # 30.29 TFLOPs for FP32 on NVIDIA L4
        torch.float16: 30.29e12,  # 30.29 TFLOPs for FP16 on NVIDIA L4
        torch.bfloat16: 30.29e12
    },
    # https://www.techpowerup.com/gpu-specs/tesla-t4.c3316
    "T4": {
        torch.float32: 8.1e12,  # 8.1 TFLOPs for FP32 on NVIDIA T4
        torch.float16: 65.13e12,  # 65.13 TFLOPs for FP16 on NVIDIA T4
        torch.bfloat16: 65.13e12
    },
    # https://www.techpowerup.com/gpu-specs/a10g.c3798
    "A10G": {
        torch.float32: 31.52e12,  # 31.52 TFLOPs for FP32 on NVIDIA A10G
        torch.float16: 31.52e12,  # 31.52 TFLOPs for FP16 on NVIDIA A10G
        torch.bfloat16: 31.52e12
    },
    # https://www.techpowerup.com/gpu-specs/a100-pcie-40-gb.c3623
    "A100": {
        torch.float32: 19.49e12,  # 19.49 TFLOPs for FP32 on NVIDIA A100
        torch.float16: 77.97e12,  # 77.97 TFLOPs for FP16 on NVIDIA A100
        torch.bfloat16: 77.97e12
    },
    # https://www.techpowerup.com/gpu-specs/geforce-rtx-3080.c3621
    "RTX_3080": {
        torch.float32: 29.77e12,  # 29.77 TFLOPs for FP32 on NVIDIA RTX 3080
        torch.float16: 29.77e12,  # 29.77 TFLOPs for FP16 on NVIDIA RTX 3080
        torch.bfloat16: 29.77e12
    },
    # https://www.techpowerup.com/gpu-specs/geforce-rtx-3090.c3622
    "RTX_3090": {
        torch.float32: 35.58e12,  # 35.58 TFLOPs for FP32 on NVIDIA RTX 3090
        torch.float16: 35.58e12,  # 35.58 TFLOPs for FP16 on NVIDIA RTX 3090
        torch.bfloat16: 35.58e12
    }
}

In [ ]:
import time

def get_gpu_model(flops_per_second_dict):
    device_name = torch.cuda.get_device_name(0)
    for model in flops_per_second_dict.keys():
        if model in device_name:
            return model
    return "Unknown"  # Default if no matching model is found


gpu_model = get_gpu_model(flops_per_second)
print("GPU Model:", gpu_model)

if gpu_model != "Unknown":

    for size in model_configs:
        print(f"\nProcessing {size}")
        config = BASE_CONFIG.copy()
        config.update(model_configs[size])

        min_batch_size = 1
        max_batch_size = None
        max_possible_batch_size = 4096

        while min_batch_size <= max_possible_batch_size:
            batch_size = (min_batch_size + max_possible_batch_size) // 2
            try:
                input_tensor = torch.randint(
                    0, config["vocab_size"],
                    (batch_size, config["context_length"]),
                    device=device
                )

                model = GPTModel(config).bfloat16().to(device)
                model.train()

                # Start timing
                torch.cuda.synchronize()
                start_time = time.time()

                # Forward & backward pass
                output = model(input_tensor)
                loss = output.sum()  # Compute a dummy loss
                loss.backward()

                # End timing
                torch.cuda.synchronize()
                end_time = time.time()

                total_time_seconds = end_time - start_time

                # Calculate FLOPs for forward pass
                macs, params = profile(model, inputs=(input_tensor,), verbose=False)
                flops_forward = 2 * macs  # Assuming one MAC equals two FLOPs

                # Estimate FLOPs for backward pass (typically 2x forward FLOPs)
                flops_backward = 2 * flops_forward

                # Total FLOPs for forward + backward passes
                total_flops = flops_forward + flops_backward  # Or total_flops = flops_forward * 3

                data_type = next(model.parameters()).dtype
                max_flops_per_second = flops_per_second[gpu_model].get(data_type, 0)

                # Compute tokens per second
                tokens_processed = batch_size * config["context_length"]
                tokens_per_second = tokens_processed / total_time_seconds

                # Compute FLOPs per token
                flops_per_token = total_flops / tokens_processed

                # Compute theoretical max tokens per second
                if flops_per_token > 0:
                    theoretical_max_tokens_per_second = max_flops_per_second / flops_per_token
                else:
                    theoretical_max_tokens_per_second = 0  # Avoid division by zero

                # Compute MFU
                if theoretical_max_tokens_per_second > 0:
                    mfu = tokens_per_second / theoretical_max_tokens_per_second
                else:
                    mfu = 0  # Avoid division by zero

                print(f"  Batch size {batch_size}: Tokens/sec: {tokens_per_second:.2f}, MFU: {mfu:.4f}")

                # If successful, try a larger batch size
                min_batch_size = batch_size + 1
                max_batch_size = batch_size

                # Clean up
                del model, input_tensor, output, loss
                torch.cuda.empty_cache()

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    # Try smaller batch size
                    max_possible_batch_size = batch_size - 1

                    # Clean up
                    try:
                        del model, input_tensor
                        torch.cuda.empty_cache()
                    except NameError:
                        pass
                else:
                    raise e

else:
    print("Unknown GPU model. Please update the flops_per_second dictionary with your GPU information.")

GPU Model: T4

Processing gpt-small (124M)

Processing gpt-medium (355M)

Processing gpt-large (774M)

Processing gpt-xl (1558M)


- a value of 1.0 is best (equal to 100%)
- Note that the batch sizes are smaller than previously because we also carry out the backward pass here, which is more memory-intensive